# Optimization Methods: Week 2 Seminar

## Convexity, constraints, KKT conditions, and dual sensitivity

This notebook accompanies `week02_seminar.tex`. Complete every **TODO** cell and add a short interpretation after each numerical experiment.


In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.optimize import minimize

rng = np.random.default_rng(42)
np.set_printoptions(precision=6, suppress=True)

## 1. SymPy essentials

SymPy stores exact symbolic expressions. Assumptions such as `real=True` or `positive=True` may change simplification results.


In [ ]:
x, y = sp.symbols("x y", real=True)
a = sp.symbols("a", positive=True)
f = sp.exp(x * y) + x**2 + sp.sin(y)

print("Expression:", f)
print("Expanded:", sp.expand((x + y) ** 3))
print("Factored:", sp.factor(x**2 - y**2))
print("Simplified:", sp.simplify((x**2 - y**2) / (x - y)))

In [ ]:
variables = sp.Matrix([x, y])
grad_f = sp.Matrix([sp.diff(f, v) for v in variables])
H_f = sp.hessian(f, variables)

display(sp.Eq(sp.Symbol("f"), f))
display(sp.Eq(sp.Symbol("nabla_f"), grad_f))
display(sp.Eq(sp.Symbol("H_f"), H_f))

In [ ]:
f_np = sp.lambdify((x, y), f, modules="numpy")
g_np = sp.lambdify((x, y), grad_f, modules="numpy")
H_np = sp.lambdify((x, y), H_f, modules="numpy")

point = (1.0, 2.0)
print("f(point) =", f_np(*point))
print("gradient =", np.asarray(g_np(*point), dtype=float).ravel())
print("Hessian =\n", np.asarray(H_np(*point), dtype=float))

### SymPy warm-up

For $q(x,y)=\tfrac12(3x^2+2xy+4y^2)-2x+y$, compute the gradient, Hessian, stationary point, and Hessian eigenvalues.


In [ ]:
# TODO: construct q, grad_q, H_q, and solve grad_q = 0.
q = None
grad_q = None
H_q = None

## 2. Task 1: convexity of sets

Decide whether each set is convex and provide an analytic proof or a counterexample. Use the numerical probe only to search for counterexamples.

1. $C_1=\{x\in\mathbb R^2:\|x\|_2\le1\}$.
2. $C_2=\{x\in\mathbb R^2:x_1x_2\ge1,\ x_1>0\}$.
3. $C_3=\{x\in\mathbb R^2:\|x\|_2\ge1\}$.
4. $C_4=\{x\in\mathbb R^3:x_i\ge0,\ \mathbf1^Tx=1\}$.


In [ ]:
def probe_convexity(sample_point, is_feasible, trials=10_000, seed=0):
    local_rng = np.random.default_rng(seed)
    for _ in range(trials):
        p, q = sample_point(local_rng), sample_point(local_rng)
        t = local_rng.random()
        z = (1.0 - t) * p + t * q
        if not is_feasible(z):
            return False, {"x": p, "y": q, "t": t, "z": z}
    return True, None


# TODO: implement samplers and feasibility checks for C1-C4.
# Passing this probe does not prove convexity. A returned counterexample proves nonconvexity.

## 3. Task 2: convexity of functions

Use symbolic Hessians and analytic reasoning. State the domain. If a function is not convex, find a point and direction $d$ such that $d^T H(x)d<0$.

1. $f_1(x,y)=x^2+4xy+5y^2$.
2. $f_2(x,y)=e^{x+y}+x^2$.
3. $f_3(x,y)=x^2y^2$.
4. $f_4(x,y)=-\log x-\log y$ on $\mathbb R_{++}^2$.


In [ ]:
functions = [
    x**2 + 4 * x * y + 5 * y**2,
    sp.exp(x + y) + x**2,
    x**2 * y**2,
    -sp.log(x) - sp.log(y),
]

# TODO: print each Hessian, analyze positive semidefiniteness, and record conclusions.
for index, expr in enumerate(functions, start=1):
    print(f"f_{index}:", expr)
    display(sp.hessian(expr, (x, y)))

## 4. Task 3: smoothness and strong convexity of a quadratic

For $f(z)=\tfrac12z^TQz+b^Tz$ with the matrix below, compute the tight constants $\mu$ and $L$, the condition number $\kappa=L/\mu$, and verify the defining inequalities on random pairs.


In [ ]:
Q = np.array([[8.0, 2.0, 0.0], [2.0, 3.0, 1.0], [0.0, 1.0, 2.0]])
b = np.array([1.0, -2.0, 0.5])

# TODO: compute eigenvalues, mu, L, and kappa.
eigenvalues = None
mu = None
L = None
kappa = None


def quadratic(z):
    return 0.5 * z @ Q @ z + b @ z


def quadratic_grad(z):
    return Q @ z + b

In [ ]:
def curvature_residuals(f, grad, mu, L, x0, x1):
    d = x1 - x0
    linear = f(x0) + grad(x0) @ d
    strong_residual = f(x1) - (linear + 0.5 * mu * np.dot(d, d))
    smooth_residual = (linear + 0.5 * L * np.dot(d, d)) - f(x1)
    return strong_residual, smooth_residual


# TODO: test at least 100 random pairs and report the minimum residuals.
# TODO: predict and verify what changes after replacing Q by 10*Q.

## 5. Task 4: equality-constrained optimization

Solve
$$\min_{x,y}(x-1)^2+2(y+1)^2 \quad\text{subject to}\quad x+2y=1.$$
Construct the Lagrangian, solve the stationarity equations, verify feasibility, and check curvature along a tangent direction.


In [ ]:
x, y, nu = sp.symbols("x y nu", real=True)
objective_eq = (x - 1) ** 2 + 2 * (y + 1) ** 2
h = x + 2 * y - 1

# TODO: construct the Lagrangian and solve its stationarity equations plus h=0.
Lagrangian_eq = None
solution_eq = None

## 6. Task 5: inequality constraints and KKT

Solve
$$\min_{x,y}(x-2)^2+(y-1)^2$$
subject to $x+y\le1$, $x\ge0$, and $y\ge0$. Write all inequalities as $g_i(x,y)\le0$. Enumerate plausible active sets, solve stationarity, and reject candidates that violate primal feasibility, dual feasibility, or complementary slackness.


In [ ]:
x, y = sp.symbols("x y", real=True)
lam1, lam2, lam3 = sp.symbols("lam1 lam2 lam3", real=True)
objective_ineq = (x - 2) ** 2 + (y - 1) ** 2
g = sp.Matrix([x + y - 1, -x, -y])
lam = sp.Matrix([lam1, lam2, lam3])
Lagrangian_ineq = objective_ineq + (lam.T * g)[0]

stationarity = [sp.diff(Lagrangian_ineq, v) for v in (x, y)]
complementarity = [lam[i] * g[i] for i in range(3)]
display(Lagrangian_ineq, stationarity, complementarity)

# TODO: solve by active-set enumeration and create a candidate table.

### Numerical KKT residuals

Implement separate residuals for stationarity, equality feasibility, inequality feasibility, dual feasibility, and complementary slackness.


In [ ]:
def kkt_residuals(grad_lagrangian, equalities, inequalities, multipliers):
    # Convention: inequalities <= 0 and multipliers >= 0.
    grad_lagrangian = np.asarray(grad_lagrangian, dtype=float)
    equalities = np.asarray(equalities, dtype=float)
    inequalities = np.asarray(inequalities, dtype=float)
    multipliers = np.asarray(multipliers, dtype=float)
    return {
        "stationarity": np.linalg.norm(grad_lagrangian),
        "equality_feasibility": np.linalg.norm(equalities),
        "inequality_feasibility": np.linalg.norm(np.maximum(inequalities, 0.0)),
        "dual_feasibility": np.linalg.norm(np.minimum(multipliers, 0.0)),
        "complementarity": np.linalg.norm(multipliers * inequalities),
    }

## 7. SciPy verification

SciPy inequality dictionaries use $c(x)\ge0$, the opposite sign from our KKT convention $g(x)\le0$. Verify the KKT solution independently.


In [ ]:
def objective_scipy(z):
    x, y = z
    return (x - 2.0) ** 2 + (y - 1.0) ** 2


constraints = [{"type": "ineq", "fun": lambda z: 1.0 - z[0] - z[1]}]
result = minimize(
    objective_scipy,
    x0=np.array([0.3, 0.3]),
    bounds=[(0.0, None), (0.0, None)],
    constraints=constraints,
    method="SLSQP",
    options={"ftol": 1e-12, "maxiter": 1000},
)
print(result)

# TODO: compare with the analytic KKT solution and report residuals.

## 8. Task 6: simplex-constrained regression

Generate $X\in\mathbb R^{20\times4}$ and $y\in\mathbb R^{20}$, then solve
$$\min_w \tfrac12\|Xw-y\|^2 \quad\text{subject to}\quad w_i\ge0,\;\mathbf1^Tw=1.$$
Identify active nonnegativity constraints, estimate multipliers from stationarity, and check complementary slackness numerically.


In [ ]:
rng = np.random.default_rng(7)
X = rng.normal(size=(20, 4))
y_data = rng.normal(size=20)


def regression_objective(w):
    residual = X @ w - y_data
    return 0.5 * residual @ residual


def regression_gradient(w):
    return X.T @ (X @ w - y_data)


# TODO: call SLSQP with bounds and the simplex equality.
# TODO: identify active coordinates using a tolerance, then estimate KKT multipliers.

## 9. Task 7: dual sensitivity

For
$$p(R)=\min_{x,y}(x-2)^2+(y-1)^2 \quad\text{subject to}\quad x+y\le R,$$
solve KKT at $R=1$, compute the multiplier $\lambda^*$, and compare $-\lambda^*$ with a central-difference estimate of $p'(1)$.


In [ ]:
def solve_resource_problem(R):
    constraint = {"type": "ineq", "fun": lambda z: R - z[0] - z[1]}
    return minimize(
        objective_scipy,
        x0=np.array([R / 2, R / 2]),
        constraints=[constraint],
        method="SLSQP",
        options={"ftol": 1e-12, "maxiter": 1000},
    )


epsilon = 1e-4
# TODO: evaluate p(1-epsilon), p(1), and p(1+epsilon).
# TODO: estimate p'(1) and compare it with -lambda_star.

## 10. Optional task: SVM and complementary slackness

Train a linear soft-margin SVM on a small two-dimensional dataset. Separate points into those outside the margin, on the margin, and inside the margin. Compare these groups with the nonzero dual coefficients and explain the result using complementary slackness.


In [ ]:
# Optional. Requires scikit-learn.
# from sklearn.datasets import make_blobs
# from sklearn.svm import SVC
# TODO: generate data, fit SVC(kernel='linear'), and inspect support vectors.

## Final checklist

Before submission, confirm that the notebook contains:

- symbolic gradients and Hessians;
- analytic proofs or counterexamples for set convexity;
- values of $\mu$, $L$, and $\kappa$;
- equality and inequality KKT systems;
- independent feasibility and KKT residual checks;
- SciPy verification;
- a numerical sensitivity estimate and its interpretation.
